
# Customer segmentation workbench built from the existing feature-engineering notebook

This notebook extends the current leakage-safe order history work into a full customer clustering pipeline.
It reuses the same prior-order fact construction, then adds:

- a finalized customer feature mart
- feature audit and transformation policy
- K-Means, DBSCAN, GMM, and Hierarchical sweeps
- internal metric tracking and stability checks
- business-readable segment profiles, labels, and actions

Two modeling principles drive the implementation:

1. **Preserve the current work**: keep the prior-order behavioral logic already built in the original notebook.
2. **Upgrade the grain**: stop at a customer-level mart instead of continuing into the user-product training matrix.

A small but important correction is applied here: basket-size features are computed from an **order-level table** before aggregating to users, which avoids item-weighting larger baskets by accident.


In [1]:

from __future__ import annotations

import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score, silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_STATE = 42
DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs/customer_segmentation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BENCHMARK_SAMPLE_N = 40000
STABILITY_SUBSAMPLES = 5
STABILITY_SAMPLE_FRAC = 0.80
SILHOUETTE_SAMPLE_N = 10000
CORRELATION_THRESHOLD = 0.92
LOWER_CLIP_Q = 0.01
UPPER_CLIP_Q = 0.99
PCA_EXPLAINED_VARIANCE = 0.90

ORDERS_DTYPES = {
    "order_id": "int32",
    "user_id": "int32",
    "eval_set": "category",
    "order_number": "int16",
    "order_dow": "int8",
    "order_hour_of_day": "int8",
    "days_since_prior_order": "float32",
}

ORDER_PRODUCTS_DTYPES = {
    "order_id": "int32",
    "product_id": "int32",
    "add_to_cart_order": "int16",
    "reordered": "int8",
}

PRODUCTS_DTYPES = {
    "product_id": "int32",
    "product_name": "string",
    "aisle_id": "int16",
    "department_id": "int8",
}

AISLES_DTYPES = {
    "aisle_id": "int16",
    "aisle": "category",
}

DEPARTMENTS_DTYPES = {
    "department_id": "int8",
    "department": "category",
}



## 1. Load data and recreate the current notebook's base tables

This section intentionally mirrors the structure of the original notebook:

- load orders and prior order items
- enrich products with aisle and department dimensions
- build a prior-order item fact table

The new pipeline then adds an **order-level table**, because customer basket metrics should be computed from orders, not from item rows.


In [2]:

orders = pd.read_csv(DATA_DIR / "orders.csv", dtype=ORDERS_DTYPES)
op_prior = pd.read_csv(DATA_DIR / "order_products__prior.csv", dtype=ORDER_PRODUCTS_DTYPES)
products = pd.read_csv(DATA_DIR / "products.csv", dtype=PRODUCTS_DTYPES)
aisles = pd.read_csv(DATA_DIR / "aisles.csv", dtype=AISLES_DTYPES)
departments = pd.read_csv(DATA_DIR / "departments.csv", dtype=DEPARTMENTS_DTYPES)

assert orders["order_id"].is_unique, "orders.csv should have one row per order_id"
assert not op_prior.duplicated(["order_id", "product_id", "add_to_cart_order"]).any(), "Duplicate prior order lines detected"
assert products["product_id"].is_unique, "products.csv should have one row per product_id"


dim_products = (
    products
    .merge(aisles, on="aisle_id", how="left", validate="many_to_one")
    .merge(departments, on="department_id", how="left", validate="many_to_one")
)

prior_orders = orders.loc[
    orders["eval_set"].eq("prior"),
    [
        "order_id",
        "user_id",
        "order_number",
        "order_dow",
        "order_hour_of_day",
        "days_since_prior_order",
    ],
].copy()

reference_orders = orders.loc[
    orders["eval_set"].isin(["train", "test"]),
    [
        "order_id",
        "user_id",
        "eval_set",
        "order_number",
        "order_dow",
        "order_hour_of_day",
        "days_since_prior_order",
    ],
].copy()

assert reference_orders["user_id"].is_unique, "Expected one final observed order per user in train/test"

prior_order_items = (
    op_prior
    .merge(prior_orders, on="order_id", how="left", validate="many_to_one")
    .merge(
        dim_products[["product_id", "aisle_id", "department_id", "aisle", "department"]],
        on="product_id",
        how="left",
        validate="many_to_one",
    )
)

prior_order_items["order_size"] = (
    prior_order_items.groupby("order_id")["product_id"].transform("size").astype("int16")
)
prior_order_items["cart_share"] = (
    prior_order_items["add_to_cart_order"].astype("float32") / prior_order_items["order_size"]
)

order_level = (
    prior_order_items
    .groupby(
        [
            "user_id",
            "order_id",
            "order_number",
            "order_dow",
            "order_hour_of_day",
            "days_since_prior_order",
        ],
        as_index=False,
    )
    .agg(
        basket_size=("product_id", "size"),
        reordered_items=("reordered", "sum"),
        avg_cart_share=("cart_share", "mean"),
        distinct_products=("product_id", "nunique"),
        distinct_departments=("department_id", "nunique"),
        distinct_aisles=("aisle_id", "nunique"),
    )
)
order_level["reorder_share"] = order_level["reordered_items"] / order_level["basket_size"]
order_level["weekend_flag"] = order_level["order_dow"].isin([0, 6]).astype("int8")
order_level["hour_angle"] = 2 * np.pi * order_level["order_hour_of_day"] / 24.0
order_level["hour_sin"] = np.sin(order_level["hour_angle"])
order_level["hour_cos"] = np.cos(order_level["hour_angle"])

pd.Series(
    {
        "users": orders["user_id"].nunique(),
        "prior_orders": len(prior_orders),
        "prior_order_items": len(prior_order_items),
        "order_level_rows": len(order_level),
        "reference_orders": len(reference_orders),
    },
    name="base_table_summary",
)


users                  206209
prior_orders          3214874
prior_order_items    32434489
order_level_rows      3008665
reference_orders       206209
Name: base_table_summary, dtype: int64


## 2. Build the customer feature mart

The feature set is centered on the behavioral variables already identified as the core clustering signals:

- frequency
- basket size
- reorder rate
- tenure
- recency
- days between orders

It then adds a small number of stronger behavior descriptors:

- time-of-day preference
- day-of-week preference
- department / aisle concentration
- basket-size consistency


In [3]:

def safe_divide(numerator, denominator, fill_value=0.0):
    numerator = np.asarray(numerator, dtype="float64")
    denominator = np.asarray(denominator, dtype="float64")
    out = np.full(numerator.shape, fill_value, dtype="float64")
    np.divide(numerator, denominator, out=out, where=denominator != 0)
    return out


def circular_hour_from_components(sin_component, cos_component):
    angle = np.arctan2(sin_component, cos_component)
    angle = np.mod(angle, 2 * np.pi)
    return 24 * angle / (2 * np.pi)


def build_concentration_features(frame, category_col, prefix):
    shares = (
        frame.groupby(["user_id", category_col])
        .size()
        .rename("events")
        .reset_index()
    )
    shares["share"] = shares["events"] / shares.groupby("user_id")["events"].transform("sum")

    concentration = (
        shares.groupby("user_id")
        .agg(
            **{
                f"{prefix}_hhi": ("share", lambda s: float(np.square(s).sum())),
                f"{prefix}_top_share": ("share", "max"),
                f"{prefix}_entropy": (
                    "share",
                    lambda s: 0.0 if len(s) <= 1 else float(-(s * np.log(s)).sum() / np.log(len(s))),
                ),
            }
        )
        .reset_index()
    )
    return concentration


def build_customer_feature_mart(prior_orders, reference_orders, prior_order_items, order_level):
    customer_order_gaps = (
        prior_orders.groupby("user_id", as_index=False)
        .agg(
            customer_prior_orders=("order_number", "max"),
            customer_avg_days_between_orders=("days_since_prior_order", "mean"),
            customer_median_days_between_orders=("days_since_prior_order", "median"),
            customer_std_days_between_orders=("days_since_prior_order", "std"),
        )
    )

    customer_order_behavior = (
        order_level.groupby("user_id", as_index=False)
        .agg(
            customer_avg_basket_size=("basket_size", "mean"),
            customer_median_basket_size=("basket_size", "median"),
            customer_std_basket_size=("basket_size", "std"),
            customer_avg_order_reorder_share=("reorder_share", "mean"),
            customer_avg_cart_share=("avg_cart_share", "mean"),
            customer_weekend_share=("weekend_flag", "mean"),
            customer_hour_sin=("hour_sin", "mean"),
            customer_hour_cos=("hour_cos", "mean"),
            customer_peak_hour_share=("order_hour_of_day", lambda s: float(s.value_counts(normalize=True).max())),
            customer_peak_dow_share=("order_dow", lambda s: float(s.value_counts(normalize=True).max())),
        )
    )

    customer_item_behavior = (
        prior_order_items.groupby("user_id", as_index=False)
        .agg(
            customer_total_items=("product_id", "size"),
            customer_distinct_products=("product_id", "nunique"),
            customer_reordered_items=("reordered", "sum"),
            customer_distinct_departments=("department_id", "nunique"),
            customer_distinct_aisles=("aisle_id", "nunique"),
        )
    )

    customer_recency = (
        reference_orders[["user_id", "days_since_prior_order"]]
        .rename(columns={"days_since_prior_order": "customer_recency_days"})
        .copy()
    )
    customer_recency["customer_recency_days"] = customer_recency["customer_recency_days"].fillna(0)

    customer_tenure = (
        prior_orders.groupby("user_id", as_index=False)
        .agg(customer_tenure_prior_days=("days_since_prior_order", "sum"))
    )
    customer_tenure["customer_tenure_prior_days"] = customer_tenure["customer_tenure_prior_days"].fillna(0)

    department_concentration = build_concentration_features(
        prior_order_items, category_col="department_id", prefix="customer_department"
    )
    aisle_concentration = build_concentration_features(
        prior_order_items, category_col="aisle_id", prefix="customer_aisle"
    )

    customer_features = (
        customer_order_gaps
        .merge(customer_order_behavior, on="user_id", how="left", validate="one_to_one")
        .merge(customer_item_behavior, on="user_id", how="left", validate="one_to_one")
        .merge(customer_recency, on="user_id", how="left", validate="one_to_one")
        .merge(customer_tenure, on="user_id", how="left", validate="one_to_one")
        .merge(department_concentration, on="user_id", how="left", validate="one_to_one")
        .merge(aisle_concentration, on="user_id", how="left", validate="one_to_one")
    )

    customer_features["customer_tenure_days"] = (
        customer_features["customer_tenure_prior_days"] + customer_features["customer_recency_days"]
    )
    customer_features["customer_reorder_rate"] = safe_divide(
        customer_features["customer_reordered_items"],
        customer_features["customer_total_items"],
        fill_value=0.0,
    )
    customer_features["customer_product_diversity"] = safe_divide(
        customer_features["customer_distinct_products"],
        customer_features["customer_total_items"],
        fill_value=0.0,
    )
    customer_features["customer_order_frequency_30d"] = 30 * safe_divide(
        customer_features["customer_prior_orders"],
        customer_features["customer_tenure_days"].clip(lower=1),
        fill_value=0.0,
    )
    customer_features["customer_basket_size_cv"] = safe_divide(
        customer_features["customer_std_basket_size"],
        customer_features["customer_avg_basket_size"],
        fill_value=0.0,
    )
    customer_features["customer_days_between_order_cv"] = safe_divide(
        customer_features["customer_std_days_between_orders"],
        customer_features["customer_avg_days_between_orders"],
        fill_value=0.0,
    )
    customer_features["customer_preferred_order_hour"] = circular_hour_from_components(
        customer_features["customer_hour_sin"],
        customer_features["customer_hour_cos"],
    )

    fill_zero_cols = [
        "customer_avg_days_between_orders",
        "customer_median_days_between_orders",
        "customer_std_days_between_orders",
        "customer_avg_basket_size",
        "customer_median_basket_size",
        "customer_std_basket_size",
        "customer_avg_order_reorder_share",
        "customer_avg_cart_share",
        "customer_weekend_share",
        "customer_hour_sin",
        "customer_hour_cos",
        "customer_peak_hour_share",
        "customer_peak_dow_share",
        "customer_department_hhi",
        "customer_department_top_share",
        "customer_department_entropy",
        "customer_aisle_hhi",
        "customer_aisle_top_share",
        "customer_aisle_entropy",
    ]
    customer_features[fill_zero_cols] = customer_features[fill_zero_cols].fillna(0)

    return customer_features


customer_features = build_customer_feature_mart(
    prior_orders=prior_orders,
    reference_orders=reference_orders,
    prior_order_items=prior_order_items,
    order_level=order_level,
)

core_snapshot = customer_features[
    [
        "user_id",
        "customer_order_frequency_30d",
        "customer_avg_basket_size",
        "customer_reorder_rate",
        "customer_tenure_days",
        "customer_recency_days",
        "customer_avg_days_between_orders",
        "customer_weekend_share",
        "customer_department_hhi",
        "customer_aisle_hhi",
    ]
].head()

display(core_snapshot)
display(customer_features.describe(include="all").T.head(20))


,user_id,customer_order_frequency_30d,customer_avg_basket_size,customer_reorder_rate,customer_tenure_days,customer_recency_days,customer_avg_days_between_orders,customer_weekend_share,customer_department_hhi,customer_aisle_hhi
0,1,1.5789,6.0000,0.6949,190.0000,14.0000,19.5556,0.0000,0.2473,0.1456
1,2,1.8421,14.0000,0.4769,228.0000,30.0000,15.2308,0.0000,0.1662,0.0981
2,3,2.5000,7.0909,0.6250,144.0000,11.0000,12.0909,0.5455,0.2645,0.1214
3,4,1.7647,3.5000,0.0556,85.0000,30.0000,13.7500,0.0000,0.1235,0.0864
4,5,2.6087,8.6667,0.3784,46.0000,6.0000,13.3333,0.3333,0.3221,0.1103


,count,mean,std,min,25%,50%,75%,max
user_id,"206,209.0000","103,105.0000","59,527.5552",1.0000,"51,553.0000","103,105.0000","154,657.0000","206,209.0000"
customer_prior_orders,"206,209.0000",15.5904,16.6548,3.0000,5.0000,9.0000,19.0000,99.0000
customer_avg_days_between_orders,"206,209.0000",15.2094,7.1053,0.0000,9.4167,14.5000,20.2857,30.0000
customer_median_days_between_orders,"206,209.0000",14.7535,8.6760,0.0000,7.0000,13.0000,21.5000,30.0000
customer_std_days_between_orders,"206,209.0000",7.3977,3.7468,0.0000,4.7310,7.7678,9.9085,21.2132
customer_avg_basket_size,"206,209.0000",9.9530,5.9756,1.0000,5.6667,8.8947,13.0000,71.6667
customer_median_basket_size,"206,209.0000",9.5781,6.1097,1.0000,5.0000,8.0000,12.5000,81.0000
customer_std_basket_size,"206,209.0000",4.1582,2.7939,0.0000,2.1249,3.6332,5.5550,55.1543
customer_avg_order_reorder_share,"206,209.0000",0.5147,0.2187,0.0000,0.3601,0.5321,0.6831,1.0000
customer_avg_cart_share,"206,209.0000",0.6026,0.0851,0.5080,0.5470,0.5729,0.6250,1.0000



## 3. Audit the feature space and finalize the clustering variables

This section does three things before any model fitting:

1. inspect skewness and outliers
2. inspect redundancy / correlation
3. define a transformation and retention policy

Core business variables are treated as protected unless they are degenerate. Optional variables can be removed when they are clearly redundant.


In [4]:

FEATURE_CONFIG = pd.DataFrame(
    [
        ("customer_order_frequency_30d", 100, True, True, True),
        ("customer_avg_basket_size", 96, True, True, True),
        ("customer_reorder_rate", 95, True, False, False),
        ("customer_tenure_days", 94, True, True, True),
        ("customer_recency_days", 93, True, True, True),
        ("customer_avg_days_between_orders", 92, True, True, True),
        ("customer_basket_size_cv", 82, False, True, True),
        ("customer_days_between_order_cv", 80, False, True, True),
        ("customer_hour_sin", 78, False, False, False),
        ("customer_hour_cos", 78, False, False, False),
        ("customer_weekend_share", 76, False, False, False),
        ("customer_department_hhi", 74, False, False, False),
        ("customer_aisle_hhi", 72, False, False, False),
        ("customer_product_diversity", 70, False, False, False),
        ("customer_peak_hour_share", 68, False, False, False),
    ],
    columns=["feature", "priority", "required", "log_eligible", "clip_eligible"],
).set_index("feature")


def audit_numeric_features(df, features):
    rows = []
    for feature in features:
        s = df[feature].astype("float64")
        q01, q25, q50, q75, q99 = s.quantile([0.01, 0.25, 0.50, 0.75, 0.99])
        iqr = q75 - q25
        lower_fence = q25 - 1.5 * iqr
        upper_fence = q75 + 1.5 * iqr
        rows.append(
            {
                "feature": feature,
                "missing_pct": float(s.isna().mean()),
                "n_unique": int(s.nunique(dropna=True)),
                "mean": float(s.mean()),
                "std": float(s.std(ddof=0)),
                "min": float(s.min()),
                "p01": float(q01),
                "median": float(q50),
                "p99": float(q99),
                "max": float(s.max()),
                "skewness": float(s.skew()),
                "lower_outlier_rate": float((s < lower_fence).mean()),
                "upper_outlier_rate": float((s > upper_fence).mean()),
            }
        )
    audit = pd.DataFrame(rows).set_index("feature")
    audit["log1p_recommended"] = (audit["skewness"] > 1.25) & (audit["min"] >= 0)
    audit["clip_recommended"] = audit["upper_outlier_rate"] > 0.01
    return audit.sort_index()


def redundant_pairs(df, features, threshold=0.92):
    corr = df[features].corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    pairs = (
        upper.stack()
        .rename("abs_corr")
        .reset_index()
        .rename(columns={"level_0": "feature_left", "level_1": "feature_right"})
        .query("abs_corr >= @threshold")
        .sort_values("abs_corr", ascending=False)
        .reset_index(drop=True)
    )
    return pairs


def build_feature_policy(feature_config, audit_table, redundant_feature_pairs):
    policy = feature_config.copy()
    policy["keep"] = True
    policy["log1p"] = audit_table["log1p_recommended"] & policy["log_eligible"]
    policy["clip"] = audit_table["clip_recommended"] & policy["clip_eligible"]

    degenerate = set(audit_table.query("n_unique <= 1 or std == 0").index)
    policy.loc[policy.index.intersection(degenerate), "keep"] = False

    protected = set(policy.index[policy["required"]])
    drops = set()

    for row in redundant_feature_pairs.itertuples(index=False):
        left = row.feature_left
        right = row.feature_right
        if left in drops or right in drops:
            continue
        if left in protected and right in protected:
            continue
        if left in protected:
            drops.add(right)
            continue
        if right in protected:
            drops.add(left)
            continue
        drop_feature = left if policy.loc[left, "priority"] < policy.loc[right, "priority"] else right
        drops.add(drop_feature)

    if drops:
        policy.loc[list(drops), "keep"] = False

    return policy


def apply_feature_policy(df, policy):
    modeled = df.copy()
    clip_bounds = {}
    keep_features = policy.index[policy["keep"]].tolist()

    for feature in keep_features:
        series = modeled[feature].astype("float64")
        if bool(policy.loc[feature, "clip"]):
            low, high = series.quantile([LOWER_CLIP_Q, UPPER_CLIP_Q])
            modeled[feature] = series.clip(low, high)
            clip_bounds[feature] = (float(low), float(high))
        if bool(policy.loc[feature, "log1p"]):
            modeled[feature] = np.log1p(modeled[feature].clip(lower=0))

    modeled[keep_features] = modeled[keep_features].replace([np.inf, -np.inf], np.nan).fillna(0)
    return modeled, clip_bounds


candidate_features = FEATURE_CONFIG.index.tolist()
audit_table = audit_numeric_features(customer_features, candidate_features)
correlated_pairs = redundant_pairs(customer_features, candidate_features, threshold=CORRELATION_THRESHOLD)
feature_policy = build_feature_policy(FEATURE_CONFIG, audit_table, correlated_pairs)

MANUAL_FORCE_KEEP = []
MANUAL_FORCE_DROP = []
if MANUAL_FORCE_KEEP:
    feature_policy.loc[MANUAL_FORCE_KEEP, "keep"] = True
if MANUAL_FORCE_DROP:
    feature_policy.loc[MANUAL_FORCE_DROP, "keep"] = False

final_features = feature_policy.index[feature_policy["keep"]].tolist()
customer_modeling, clip_bounds = apply_feature_policy(
    customer_features[["user_id"] + candidate_features],
    feature_policy,
)

print("Final clustering features:", final_features)
display(audit_table)
display(correlated_pairs.head(20))
display(feature_policy)


Final clustering features: ['customer_order_frequency_30d', 'customer_avg_basket_size', 'customer_reorder_rate', 'customer_tenure_days', 'customer_recency_days', 'customer_avg_days_between_orders', 'customer_basket_size_cv', 'customer_days_between_order_cv', 'customer_hour_sin', 'customer_hour_cos', 'customer_weekend_share', 'customer_department_hhi', 'customer_aisle_hhi', 'customer_peak_hour_share']


,missing_pct,n_unique,mean,std,min,p01,median,p99,max,skewness,lower_outlier_rate,upper_outlier_rate,log1p_recommended,clip_recommended
feature,,,,,,,,,,,,,,
customer_aisle_hhi,0.0000,117657,0.1141,0.0995,0.0198,0.0325,0.0860,0.5511,1.0000,4.2720,0.0000,0.0747,True,True
customer_avg_basket_size,0.0000,18303,9.9530,5.9756,1.0000,1.3333,8.8947,29.0000,71.6667,1.2565,0.0000,0.0277,True,True
customer_avg_days_between_orders,0.0000,8628,15.2094,7.1053,0.0000,3.0000,14.5000,30.0000,30.0000,0.3433,0.0000,0.0000,False,False
customer_basket_size_cv,0.0000,124092,0.4476,0.2108,0.0000,0.0000,0.4314,1.0815,2.0911,0.6925,0.0000,0.0251,False,True
customer_days_between_order_cv,0.0000,136566,0.5643,0.2750,0.0000,0.0000,0.5720,1.3061,3.8437,0.1914,0.0000,0.0158,False,True
customer_department_hhi,0.0000,129633,0.2365,0.1337,0.0692,0.0970,0.1986,0.8025,1.0000,2.5017,0.0000,0.0613,True,True
customer_hour_cos,0.0000,102178,-0.4975,0.2995,-1.0000,-0.9659,-0.5471,0.4353,1.0000,1.0505,0.0000,0.0248,False,True
customer_hour_sin,0.0000,113107,-0.2254,0.3602,-1.0000,-0.9495,-0.2415,0.7071,1.0000,0.2981,0.0000,0.0085,False,False
customer_order_frequency_30d,0.0000,8564,2.5795,2.1549,1.0000,1.0000,2.0000,9.3999,180.0000,17.0923,0.0000,0.0586,True,True


,feature_left,feature_right,abs_corr
0,customer_reorder_rate,customer_product_diversity,1.0000


,priority,required,log_eligible,clip_eligible,keep,log1p,clip
feature,,,,,,,
customer_order_frequency_30d,100,True,True,True,True,True,True
customer_avg_basket_size,96,True,True,True,True,True,True
customer_reorder_rate,95,True,False,False,True,False,False
customer_tenure_days,94,True,True,True,True,False,False
customer_recency_days,93,True,True,True,True,False,False
customer_avg_days_between_orders,92,True,True,True,True,False,False
customer_basket_size_cv,82,False,True,True,True,False,True
customer_days_between_order_cv,80,False,True,True,True,False,True
customer_hour_sin,78,False,False,False,True,False,False



## 4. Standardize the final features and create the PCA view

The comparison is done on two representations of the same finalized feature set:

- **scaled**: the full standardized customer feature set
- **pca**: a reduced representation that preserves most variance while testing whether dimensionality reduction helps quality or just speed

For fair model comparison, all algorithms are benchmarked on the same customer sample if the full user base is large.


In [5]:

scaler = StandardScaler()
X_full_scaled = scaler.fit_transform(customer_modeling[final_features])

pca = PCA(n_components=PCA_EXPLAINED_VARIANCE, svd_solver="full", random_state=RANDOM_STATE)
X_full_pca = pca.fit_transform(X_full_scaled)

n_customers = len(customer_modeling)
benchmark_n = min(BENCHMARK_SAMPLE_N, n_customers)
rng = np.random.default_rng(RANDOM_STATE)
benchmark_idx = np.sort(rng.choice(n_customers, size=benchmark_n, replace=False))

X_views_full = {
    "scaled": X_full_scaled,
    "pca": X_full_pca,
}
X_views_benchmark = {
    name: X[benchmark_idx] for name, X in X_views_full.items()
}

benchmark_users = customer_modeling.iloc[benchmark_idx][["user_id"]].reset_index(drop=True)

pca_summary = pd.Series(
    {
        "n_customers": n_customers,
        "benchmark_n": benchmark_n,
        "scaled_dimensions": X_full_scaled.shape[1],
        "pca_dimensions": X_full_pca.shape[1],
        "pca_explained_variance": float(pca.explained_variance_ratio_.sum()),
    },
    name="matrix_summary",
)

display(pca_summary)


n_customers              206,209.0000
benchmark_n               40,000.0000
scaled_dimensions             14.0000
pca_dimensions                 9.0000
pca_explained_variance         0.9070
Name: matrix_summary, dtype: float64


## 5. Clustering utilities and parameter sweeps

A few implementation details matter here:

- K-Means is the baseline and stores SSE for elbow analysis.
- GMM stores AIC and BIC as supplementary diagnostics.
- DBSCAN uses an adaptive `eps` grid derived from k-nearest-neighbor distances instead of arbitrary fixed values.
- Silhouette and Davies-Bouldin are calculated consistently, with DBSCAN scored on the non-noise subset only.


In [6]:

import matplotlib.pyplot as plt


def serialize_params(params):
    return json.dumps(params, sort_keys=True)


def count_clusters(labels):
    labels = np.asarray(labels)
    return int(np.unique(labels[labels != -1]).size)


def evaluate_partition(X, labels, extra_metrics=None):
    labels = np.asarray(labels)
    non_noise_mask = labels != -1
    non_noise_labels = labels[non_noise_mask]
    n_clusters = count_clusters(labels)
    noise_share = float((labels == -1).mean())
    usable_n = int(non_noise_mask.sum())

    non_noise_counts = pd.Series(non_noise_labels).value_counts(normalize=True) if usable_n else pd.Series(dtype=float)
    largest_cluster_share = float(non_noise_counts.max()) if len(non_noise_counts) else np.nan
    smallest_cluster_share = float(non_noise_counts.min()) if len(non_noise_counts) else np.nan
    balance_entropy = (
        float(-(non_noise_counts * np.log(non_noise_counts)).sum() / np.log(len(non_noise_counts)))
        if len(non_noise_counts) > 1 else np.nan
    )

    if n_clusters >= 2 and usable_n > n_clusters:
        silhouette_n = min(SILHOUETTE_SAMPLE_N, usable_n)
        silhouette = float(
            silhouette_score(
                X[non_noise_mask],
                non_noise_labels,
                sample_size=silhouette_n,
                random_state=RANDOM_STATE,
            )
        )
        dbi = float(davies_bouldin_score(X[non_noise_mask], non_noise_labels))
        is_valid = True
    else:
        silhouette = np.nan
        dbi = np.nan
        is_valid = False

    metrics = {
        "is_valid": is_valid,
        "n_clusters": n_clusters,
        "noise_share": noise_share,
        "largest_cluster_share": largest_cluster_share,
        "smallest_cluster_share": smallest_cluster_share,
        "cluster_balance_entropy": balance_entropy,
        "silhouette_score": silhouette,
        "davies_bouldin_index": dbi,
    }
    if extra_metrics:
        metrics.update(extra_metrics)
    return metrics


def fit_predict_model(model_name, X, params, random_state=RANDOM_STATE):
    if model_name == "kmeans":
        estimator = KMeans(
            n_clusters=params["n_clusters"],
            n_init=25,
            random_state=random_state,
        )
        labels = estimator.fit_predict(X)
        extra = {"sse": float(estimator.inertia_), "aic": np.nan, "bic": np.nan}
        return estimator, labels, extra

    if model_name == "gmm":
        estimator = GaussianMixture(
            n_components=params["n_components"],
            covariance_type=params["covariance_type"],
            n_init=3,
            reg_covar=1e-6,
            random_state=random_state,
        )
        labels = estimator.fit_predict(X)
        extra = {
            "sse": np.nan,
            "aic": float(estimator.aic(X)),
            "bic": float(estimator.bic(X)),
        }
        return estimator, labels, extra

    if model_name == "agglomerative":
        kwargs = {
            "n_clusters": params["n_clusters"],
            "linkage": params["linkage"],
        }
        if params["linkage"] != "ward":
            kwargs["metric"] = params.get("metric", "euclidean")
        try:
            estimator = AgglomerativeClustering(**kwargs)
        except TypeError:
            if "metric" in kwargs:
                kwargs["affinity"] = kwargs.pop("metric")
            estimator = AgglomerativeClustering(**kwargs)
        labels = estimator.fit_predict(X)
        extra = {"sse": np.nan, "aic": np.nan, "bic": np.nan}
        return estimator, labels, extra

    if model_name == "dbscan":
        estimator = DBSCAN(
            eps=params["eps"],
            min_samples=params["min_samples"],
            n_jobs=-1,
        )
        labels = estimator.fit_predict(X)
        extra = {"sse": np.nan, "aic": np.nan, "bic": np.nan}
        return estimator, labels, extra

    raise ValueError(f"Unsupported model: {model_name}")


def build_dbscan_grid(X, min_samples_values=(5, 10, 15), quantiles=(0.90, 0.92, 0.94, 0.96, 0.98)):
    grid = []
    for min_samples in min_samples_values:
        nn = NearestNeighbors(n_neighbors=min_samples)
        nn.fit(X)
        distances, _ = nn.kneighbors(X)
        kth_distance = np.sort(distances[:, -1])
        eps_values = np.unique(np.round(np.quantile(kth_distance, quantiles), 4))
        for eps in eps_values:
            grid.append({"eps": float(eps), "min_samples": int(min_samples)})
    return grid


def build_model_specs(X):
    specs = []
    specs.extend(("kmeans", {"n_clusters": k}) for k in range(2, 11))
    specs.extend(
        ("gmm", {"n_components": k, "covariance_type": covariance_type})
        for k in range(2, 11)
        for covariance_type in ["full", "diag"]
    )
    specs.extend(
        ("agglomerative", {"n_clusters": k, "linkage": linkage, "metric": "euclidean"})
        for k in range(2, 11)
        for linkage in ["ward", "complete", "average"]
    )
    specs.extend(("dbscan", params) for params in build_dbscan_grid(X))
    return specs


def run_model_sweep(X_views):
    results = []
    label_store = {}

    for view_name, X in X_views.items():
        for model_name, params in build_model_specs(X):
            params_json = serialize_params(params)
            start = time.perf_counter()
            estimator, labels, extra = fit_predict_model(model_name, X, params, random_state=RANDOM_STATE)
            runtime_seconds = time.perf_counter() - start
            metrics = evaluate_partition(X, labels, extra_metrics=extra)
            result_row = {
                "view": view_name,
                "model": model_name,
                "params_json": params_json,
                "runtime_seconds": runtime_seconds,
                **metrics,
            }
            results.append(result_row)
            label_store[(view_name, model_name, params_json)] = labels

    return pd.DataFrame(results), label_store


model_results, label_store = run_model_sweep(X_views_benchmark)
valid_model_results = model_results.loc[model_results["is_valid"]].copy()

kmeans_elbow = valid_model_results.loc[valid_model_results["model"].eq("kmeans")].copy()

fig, ax = plt.subplots(figsize=(8, 4))
for view_name, frame in kmeans_elbow.groupby("view"):
    n_clusters = frame["params_json"].map(lambda x: json.loads(x)["n_clusters"])
    ax.plot(n_clusters, frame["sse"], marker="o", label=view_name)
ax.set_title("K-Means elbow / SSE by k")
ax.set_xlabel("k")
ax.set_ylabel("SSE")
ax.legend()
plt.show()

leaderboard = (
    valid_model_results
    .sort_values(
        ["silhouette_score", "davies_bouldin_index", "runtime_seconds"],
        ascending=[False, True, True],
    )
    .reset_index(drop=True)
)

display(leaderboard.head(20))


KeyboardInterrupt: 


## 6. Stability checks and formal model selection

A single run is not enough. The shortlist is stress-tested in two ways:

- **subsample stability** for every shortlisted model
- **seed stability** for seed-sensitive models (K-Means and GMM)

Final selection combines:

- statistical quality
- stability
- business interpretability
- deployment practicality


In [ ]:


def shortlist_candidates(results, top_n_per_group=3):
    filtered = results.loc[
        results["is_valid"]
        & results["n_clusters"].between(2, 10)
        & (results["largest_cluster_share"] <= 0.80)
    ].copy()

    shortlisted = (
        filtered.sort_values(
            ["silhouette_score", "davies_bouldin_index", "runtime_seconds"],
            ascending=[False, True, True],
        )
        .groupby(["view", "model"], group_keys=False)
        .head(top_n_per_group)
        .reset_index(drop=True)
    )
    return shortlisted


def subsample_stability(model_name, X, params, base_labels, n_iter=STABILITY_SUBSAMPLES, sample_frac=STABILITY_SAMPLE_FRAC):
    rng = np.random.default_rng(RANDOM_STATE)
    ari_scores = []
    valid_runs = 0

    for iteration in range(n_iter):
        sample_size = int(len(X) * sample_frac)
        sample_idx = np.sort(rng.choice(len(X), size=sample_size, replace=False))
        _, sampled_labels, _ = fit_predict_model(
            model_name,
            X[sample_idx],
            params,
            random_state=RANDOM_STATE + iteration + 1,
        )
        base_subset = np.asarray(base_labels)[sample_idx]
        if count_clusters(base_subset) < 2 or count_clusters(sampled_labels) < 2:
            continue
        valid_runs += 1
        ari_scores.append(adjusted_rand_score(base_subset, sampled_labels))

    return {
        "subsample_stability_mean_ari": float(np.mean(ari_scores)) if ari_scores else np.nan,
        "subsample_stability_median_ari": float(np.median(ari_scores)) if ari_scores else np.nan,
        "subsample_stability_valid_runs": valid_runs,
    }


def seed_stability(model_name, X, params, base_labels, seeds=(7, 11, 23, 59)):
    if model_name not in {"kmeans", "gmm"}:
        return {
            "seed_stability_mean_ari": np.nan,
            "seed_stability_median_ari": np.nan,
            "seed_stability_valid_runs": 0,
        }

    ari_scores = []
    valid_runs = 0
    for seed in seeds:
        _, alt_labels, _ = fit_predict_model(model_name, X, params, random_state=seed)
        if count_clusters(alt_labels) < 2:
            continue
        valid_runs += 1
        ari_scores.append(adjusted_rand_score(base_labels, alt_labels))

    return {
        "seed_stability_mean_ari": float(np.mean(ari_scores)) if ari_scores else np.nan,
        "seed_stability_median_ari": float(np.median(ari_scores)) if ari_scores else np.nan,
        "seed_stability_valid_runs": valid_runs,
    }


def business_interpretability_score(row):
    cluster_score = 1.0 if 3 <= row["n_clusters"] <= 7 else 0.6 if 2 <= row["n_clusters"] <= 8 else 0.2
    dominance_score = 1.0 if row["largest_cluster_share"] <= 0.50 else max(0.0, 1 - (row["largest_cluster_share"] - 0.50) / 0.50)
    tail_score = 1.0 if row["smallest_cluster_share"] >= 0.05 else min(1.0, row["smallest_cluster_share"] / 0.05)
    noise_score = 1.0 if row["noise_share"] <= 0.10 else max(0.0, 1 - (row["noise_share"] - 0.10) / 0.40)
    deployability_score = 1.0 if row["model"] in {"kmeans", "gmm"} else 0.6
    return np.mean([cluster_score, dominance_score, tail_score, noise_score, deployability_score])


shortlisted = shortlist_candidates(valid_model_results, top_n_per_group=3)

stability_rows = []
for row in shortlisted.itertuples(index=False):
    params = json.loads(row.params_json)
    X = X_views_benchmark[row.view]
    base_labels = label_store[(row.view, row.model, row.params_json)]

    stability_row = {
        "view": row.view,
        "model": row.model,
        "params_json": row.params_json,
        **subsample_stability(row.model, X, params, base_labels),
        **seed_stability(row.model, X, params, base_labels),
    }
    stability_rows.append(stability_row)

stability_results = pd.DataFrame(stability_rows)
selection_table = valid_model_results.merge(
    stability_results,
    on=["view", "model", "params_json"],
    how="left",
)
selection_table["stability_score"] = selection_table[
    ["subsample_stability_mean_ari", "seed_stability_mean_ari"]
].mean(axis=1, skipna=True)
selection_table["statistical_score"] = (
    selection_table["silhouette_score"].rank(pct=True)
    + (-selection_table["davies_bouldin_index"]).rank(pct=True)
    + selection_table["stability_score"].rank(pct=True, na_option="bottom")
) / 3
selection_table["business_score"] = selection_table.apply(business_interpretability_score, axis=1)
selection_table["composite_score"] = 0.60 * selection_table["statistical_score"] + 0.40 * selection_table["business_score"]

final_model_row = (
    selection_table
    .sort_values(["composite_score", "silhouette_score", "davies_bouldin_index"], ascending=[False, False, True])
    .iloc[0]
)

selection_columns = [
    "view",
    "model",
    "params_json",
    "n_clusters",
    "silhouette_score",
    "davies_bouldin_index",
    "stability_score",
    "business_score",
    "composite_score",
    "noise_share",
    "largest_cluster_share",
    "runtime_seconds",
]

display(selection_table.sort_values("composite_score", ascending=False)[selection_columns].head(15))
print("Selected final model:")
display(final_model_row[selection_columns])



## 7. Fit the final model on the full customer base

The chosen configuration is refit on the complete transformed customer matrix, not only on the benchmark sample.
Then the cluster assignments are converted into business-readable segments.


In [ ]:

final_model_name = final_model_row["model"]
final_view = final_model_row["view"]
final_params = json.loads(final_model_row["params_json"])

final_estimator, final_labels, final_extra = fit_predict_model(
    final_model_name,
    X_views_full[final_view],
    final_params,
    random_state=RANDOM_STATE,
)

customer_segments = customer_features[["user_id"]].copy()
customer_segments["cluster_id"] = final_labels.astype("int32")
customer_segments["model_view"] = final_view
customer_segments["model_name"] = final_model_name

customer_segment_base = customer_features.merge(customer_segments[["user_id", "cluster_id"]], on="user_id", how="left")

profile_metrics = [
    "customer_order_frequency_30d",
    "customer_avg_basket_size",
    "customer_reorder_rate",
    "customer_tenure_days",
    "customer_recency_days",
    "customer_avg_days_between_orders",
    "customer_weekend_share",
    "customer_hour_sin",
    "customer_hour_cos",
    "customer_department_hhi",
    "customer_aisle_hhi",
    "customer_basket_size_cv",
]

overall_means = customer_segment_base[profile_metrics].mean()
overall_stds = customer_segment_base[profile_metrics].std(ddof=0).replace(0, 1)

cluster_profile = (
    customer_segment_base.groupby("cluster_id", as_index=False)
    .agg(
        customers=("user_id", "size"),
        avg_order_frequency_30d=("customer_order_frequency_30d", "mean"),
        avg_basket_size=("customer_avg_basket_size", "mean"),
        avg_reorder_rate=("customer_reorder_rate", "mean"),
        avg_tenure_days=("customer_tenure_days", "mean"),
        avg_recency_days=("customer_recency_days", "mean"),
        avg_days_between_orders=("customer_avg_days_between_orders", "mean"),
        avg_weekend_share=("customer_weekend_share", "mean"),
        avg_hour_sin=("customer_hour_sin", "mean"),
        avg_hour_cos=("customer_hour_cos", "mean"),
        avg_department_hhi=("customer_department_hhi", "mean"),
        avg_aisle_hhi=("customer_aisle_hhi", "mean"),
        avg_basket_size_cv=("customer_basket_size_cv", "mean"),
    )
)
cluster_profile["share_of_customers"] = cluster_profile["customers"] / cluster_profile["customers"].sum()
cluster_profile["preferred_order_hour"] = circular_hour_from_components(
    cluster_profile["avg_hour_sin"],
    cluster_profile["avg_hour_cos"],
)
cluster_profile["preferred_daypart"] = pd.cut(
    cluster_profile["preferred_order_hour"],
    bins=[-0.1, 6, 11, 14, 18, 24],
    labels=["Overnight", "Morning", "Midday", "Afternoon", "Evening"],
).astype("string")

profile_for_z = cluster_profile[[
    "avg_order_frequency_30d",
    "avg_basket_size",
    "avg_reorder_rate",
    "avg_tenure_days",
    "avg_recency_days",
    "avg_days_between_orders",
    "avg_department_hhi",
    "avg_aisle_hhi",
    "avg_basket_size_cv",
]].copy()
profile_for_z.columns = [
    "customer_order_frequency_30d",
    "customer_avg_basket_size",
    "customer_reorder_rate",
    "customer_tenure_days",
    "customer_recency_days",
    "customer_avg_days_between_orders",
    "customer_department_hhi",
    "customer_aisle_hhi",
    "customer_basket_size_cv",
]
profile_z = (profile_for_z - overall_means[profile_for_z.columns]) / overall_stds[profile_for_z.columns]
profile_z.columns = [f"{col}_z" for col in profile_z.columns]
cluster_profile = pd.concat([cluster_profile, profile_z], axis=1)


def assign_segment_label(row):
    if row["cluster_id"] == -1:
        return "Atypical / unassigned shoppers"
    if row["customer_tenure_days_z"] < -0.75 and row["customer_reorder_rate_z"] < -0.50:
        return "New low-commitment users"
    if row["customer_order_frequency_30d_z"] > 0.75 and row["customer_reorder_rate_z"] > 0.50 and row["customer_recency_days_z"] < 0.25:
        return "High-frequency loyal buyers"
    if row["customer_avg_basket_size_z"] > 0.75 and row["customer_order_frequency_30d_z"] < 0.00:
        return "Infrequent bulk shoppers"
    if row["customer_recency_days_z"] > 0.75 and row["customer_tenure_days_z"] > 0.00 and row["customer_order_frequency_30d_z"] < 0.25:
        return "At-risk lapsed regulars"
    if max(row["customer_department_hhi_z"], row["customer_aisle_hhi_z"]) > 0.75:
        return "Category-focused routine shoppers"
    if row["customer_order_frequency_30d_z"] > 0.25 and row["customer_avg_basket_size_z"] > 0.25:
        return "Engaged steady households"
    return "General convenience shoppers"


SEGMENT_ACTIONS = {
    "High-frequency loyal buyers": "Prioritize retention, VIP rewards, and adjacent-category cross-sell on frequently repeated items.",
    "Infrequent bulk shoppers": "Use stock-up timing nudges, bundle promotions, and larger-pack recommendations around their reorder window.",
    "New low-commitment users": "Run onboarding sequences, first-to-second-order incentives, and light-friction discovery offers.",
    "At-risk lapsed regulars": "Deploy win-back reminders, personalized coupons, and reorder prompts for historically repeated products.",
    "Category-focused routine shoppers": "Cross-sell adjacent aisles and complementary products while preserving their mission-driven basket.",
    "Engaged steady households": "Use loyalty-building offers and basket-expansion campaigns to increase category breadth.",
    "General convenience shoppers": "Promote convenience bundles, quick-reorder paths, and curated essentials to reduce shopping friction.",
    "Atypical / unassigned shoppers": "Treat as a monitoring cohort and inspect separately before activating broad marketing programs.",
}

cluster_profile["segment_label"] = cluster_profile.apply(assign_segment_label, axis=1)
cluster_profile["recommended_action"] = cluster_profile["segment_label"].map(SEGMENT_ACTIONS)

segment_lookup = cluster_profile[["cluster_id", "segment_label", "recommended_action"]]
customer_segments = customer_segments.merge(segment_lookup, on="cluster_id", how="left")

profile_columns_to_show = [
    "cluster_id",
    "segment_label",
    "customers",
    "share_of_customers",
    "avg_order_frequency_30d",
    "avg_basket_size",
    "avg_reorder_rate",
    "avg_tenure_days",
    "avg_recency_days",
    "avg_days_between_orders",
    "preferred_order_hour",
    "preferred_daypart",
    "recommended_action",
]

display(cluster_profile[profile_columns_to_show].sort_values("customers", ascending=False))



## 8. Export the artifacts

The notebook writes out everything needed for downstream review:

- finalized customer feature mart
- feature policy and audit
- full model evaluation table
- final customer-to-segment assignments
- cluster profile table


In [ ]:

audit_table.reset_index().to_csv(OUTPUT_DIR / "feature_audit.csv", index=False)
feature_policy.reset_index().to_csv(OUTPUT_DIR / "feature_policy.csv", index=False)
model_results.to_csv(OUTPUT_DIR / "clustering_model_results.csv", index=False)
selection_table.to_csv(OUTPUT_DIR / "clustering_model_selection.csv", index=False)
customer_features.to_csv(OUTPUT_DIR / "customer_feature_mart.csv", index=False)
customer_segments.to_csv(OUTPUT_DIR / "customer_segments.csv", index=False)
cluster_profile.to_csv(OUTPUT_DIR / "cluster_profile_table.csv", index=False)

print(f"Artifacts saved to: {OUTPUT_DIR.resolve()}")
